# LLM tool calling

Harness Agent tutorial — `ch01_llm_tool_calling.ipynb`


## Chapter objectives

- Explain **function calling** / tool use vs plain text completions.
- Trace one request: schema → model → `tool_calls` → execution → tool message → final answer.
- Call `complete_with_tools()` via the OpenAI-compatible provider.


## Prerequisites

ch00; `pip install -e .`; API key recommended for live cells.


## Concept: Function calling

Models can return **structured tool calls** instead of answering directly. The harness:

1. Sends tool **schemas** (JSON Schema) with the conversation.
2. Receives `tool_calls` with a name and arguments.
3. Executes the tool and appends a `tool` role message with the observation.
4. Calls the model again until it returns user-facing text.

Hermes, OpenClaw, and IDE agents all rely on this pattern; differences are in registry size, observation format, and sandboxing.


## How it works

```mermaid
sequenceDiagram
  participant H as Harness
  participant M as Model
  participant T as Tool
  H->>M: messages + tools[]
  M->>H: tool_calls
  H->>T: dispatch(name, args)
  T->>H: observation JSON
  H->>M: role=tool content
  M->>H: assistant text
```

Implementation: `harness_agent/providers/base.py` defines the contract; `openai_compat.py` and `anthropic_compat.py` adapt vendor APIs.


## Reference implementation map

| Harness Agent | Nous Research agent | OpenClaw |
|---------------|---------------------|----------|
| `providers/openai_compat.py` | `model_tools.py` | tool definitions in workspace |
| `providers/base.py` | provider adapters | model routing |

External path example: `model_tools.py` collects schemas from the tool registry.


## Design choices in harness_agent

Providers are lazy-initialized so `harness-agent doctor` and cron listing work without API keys until a completion runs.


## Implementation walkthrough


In [ ]:
from harness_agent.providers.openai_compat import OpenAIProvider
from harness_agent.types import Message
print('Set OPENAI_API_KEY then uncomment:')
# prov = OpenAIProvider()
# print(prov.complete_with_tools([Message(role='user', content='2+2?')], [], model='gpt-4o-mini'))


## Trace one request


In [ ]:
print('Trace: schemas + messages -> API -> tool_calls or text')


## Hands-on exercise

1. Ensure `HARNESS_AGENT_HOME` points to `labs/`.
2. Run the implementation cells below.
3. Try the matching `harness-agent` command from README for **ch01_llm_tool_calling**.


## Common pitfalls

- Missing API keys for live cells.
- Passing invalid JSON Schema (models reject or ignore tools).
- Forgetting the second model call after tool results.


## Checkpoint questions

1. What message role carries tool output back to the model?
2. When does the model return `tool_calls` vs plain text?
3. Where is the provider contract defined in Harness Agent?


## Summary & next chapter

**ch02** unifies **messages** and multiple **provider** shapes.
